In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName("oulad-etl")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

HDFS = "hdfs://namenode:9000/data/oulad"

df = spark.read.option("header", True).csv(f"{HDFS}/raw/csv/studentInfo.csv")
print(df.count())
df.show(5)

32593
+-----------+-----------------+----------+------+--------------------+--------------------+--------+--------+--------------------+---------------+----------+------------+
|code_module|code_presentation|id_student|gender|              region|   highest_education|imd_band|age_band|num_of_prev_attempts|studied_credits|disability|final_result|
+-----------+-----------------+----------+------+--------------------+--------------------+--------+--------+--------------------+---------------+----------+------------+
|        AAA|            2013J|     11391|     M| East Anglian Region|    HE Qualification| 90-100%|    55<=|                   0|            240|         N|        Pass|
|        AAA|            2013J|     28400|     F|            Scotland|    HE Qualification|  20-30%|   35-55|                   0|             60|         N|        Pass|
|        AAA|            2013J|     30268|     F|North Western Region|A Level or Equiva...|  30-40%|   35-55|                   0|         

In [2]:
schema_vle_log = StructType([
    StructField("code_module", StringType()),
    StructField("code_presentation", StringType()),
    StructField("id_student", IntegerType()),
    StructField("id_site", IntegerType()),
    StructField("date", IntegerType()),
    StructField("sum_click", IntegerType()),
])

# CSV -> JSON Lines di HDFS
(spark.read.option("header", True).schema(schema_vle_log)
      .csv(f"{HDFS}/raw/csv/studentVle.csv")
      .write.mode("overwrite").json(f"{HDFS}/raw/json/activityLog"))

# Baca kembali dari JSON untuk verifikasi
log = spark.read.schema(schema_vle_log).json(f"{HDFS}/raw/json/activityLog")
print(log.count())
log.show(5)

10655280
+-----------+-----------------+----------+-------+----+---------+
|code_module|code_presentation|id_student|id_site|date|sum_click|
+-----------+-----------------+----------+-------+----+---------+
|        AAA|            2013J|     28400| 546652| -10|        4|
|        AAA|            2013J|     28400| 546652| -10|        1|
|        AAA|            2013J|     28400| 546652| -10|        1|
|        AAA|            2013J|     28400| 546614| -10|       11|
|        AAA|            2013J|     28400| 546714| -10|        1|
+-----------+-----------------+----------+-------+----+---------+
only showing top 5 rows



In [3]:
schema_vle_log = StructType([
    StructField("code_module", StringType()),
    StructField("code_presentation", StringType()),
    StructField("id_student", IntegerType()),
    StructField("id_site", IntegerType()),
    StructField("date", IntegerType()),
    StructField("sum_click", IntegerType()),
])

log = spark.read.schema(schema_vle_log).json(f"{HDFS}/raw/json/activityLog")
print(log.count())
log.show(5)

10655280
+-----------+-----------------+----------+-------+----+---------+
|code_module|code_presentation|id_student|id_site|date|sum_click|
+-----------+-----------------+----------+-------+----+---------+
|        AAA|            2013J|     28400| 546652| -10|        4|
|        AAA|            2013J|     28400| 546652| -10|        1|
|        AAA|            2013J|     28400| 546652| -10|        1|
|        AAA|            2013J|     28400| 546614| -10|       11|
|        AAA|            2013J|     28400| 546714| -10|        1|
+-----------+-----------------+----------+-------+----+---------+
only showing top 5 rows



In [4]:
def read_csv(name):
    return (spark.read.option("header", True)
            .option("inferSchema", True)
            .csv(f"{HDFS}/raw/csv/{name}.csv"))

student_info       = read_csv("studentInfo")
student_reg        = read_csv("studentRegistration")
student_assessment = read_csv("studentAssessment")
assessments        = read_csv("assessments")
vle                = read_csv("vle")
courses            = read_csv("courses")

for name, d in [("studentInfo", student_info), ("studentRegistration", student_reg),
                ("studentAssessment", student_assessment), ("assessments", assessments),
                ("vle", vle), ("courses", courses)]:
    print(name, d.count())

studentInfo 32593
studentRegistration 32593
studentAssessment 173912
assessments 206
vle 6364
courses 22


In [5]:
# Jumlah null per kolom
log.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in log.columns]).show()

# Statistik date dan sum_click
log.describe("date", "sum_click").show()

# Nilai sum_click yang tidak wajar
print("sum_click <= 0:", log.filter(F.col("sum_click") <= 0).count())

# Baris yang kuncinya sama persis (siswa, situs, hari yang sama)
key = ["code_module", "code_presentation", "id_student", "id_site", "date"]
print("baris dengan kunci ganda:", log.count() - log.dropDuplicates(key).count())

+-----------+-----------------+----------+-------+----+---------+
|code_module|code_presentation|id_student|id_site|date|sum_click|
+-----------+-----------------+----------+-------+----+---------+
|          0|                0|         0|      0|   0|        0|
+-----------+-----------------+----------+-------+----+---------+

+-------+-----------------+------------------+
|summary|             date|         sum_click|
+-------+-----------------+------------------+
|  count|         10655280|          10655280|
|   mean|95.17399955702713|3.7169458709672574|
| stddev|76.07130084050435| 8.849046655101391|
|    min|              -25|                 1|
|    max|              269|              6977|
+-------+-----------------+------------------+

sum_click <= 0: 0
baris dengan kunci ganda: 2195960


In [6]:
from pyspark.sql.functions import broadcast

# Batas hari aktivitas yang dipakai sebagai fitur.
# None = seluruh kursus. Isi misalnya 60 jika nanti ingin prediksi dini.
CUTOFF = None

keys = ["code_module", "code_presentation", "id_student"]
log_f = log if CUTOFF is None else log.filter(F.col("date") <= CUTOFF)

# 1) Fitur ringkasan aktivitas
vle_agg = (log_f.groupBy(*keys).agg(
    F.sum("sum_click").alias("total_click"),
    F.countDistinct("date").alias("active_days"),
    F.countDistinct("id_site").alias("distinct_sites"),
    F.min("date").alias("first_activity_day"),
    F.max("date").alias("last_activity_day"),
).withColumn("avg_click_per_active_day",
             F.round(F.col("total_click") / F.col("active_days"), 3)))

# 2) Klik per jenis aktivitas (join dengan vle, tabel kecil -> broadcast)
vle_types = vle.select("id_site", "code_module", "code_presentation", "activity_type")
log_act = log_f.join(broadcast(vle_types),
                     ["id_site", "code_module", "code_presentation"], "left")
print("baris tanpa activity_type:", log_act.filter(F.col("activity_type").isNull()).count())

act_agg = (log_act.groupBy(*keys).pivot("activity_type")
           .agg(F.sum("sum_click")).fillna(0))
for c in act_agg.columns:
    if c not in keys:
        act_agg = act_agg.withColumnRenamed(c, f"click_{c}")

# 3) Gabungkan keduanya
vle_features = vle_agg.join(act_agg, keys, "inner")
print("jumlah baris:", vle_features.count())
print("jumlah kolom:", len(vle_features.columns))
vle_features.show(5, truncate=False)

# 4) Simpan ke HDFS (Parquet)
vle_features.write.mode("overwrite").parquet(f"{HDFS}/processed/vle_features")

baris tanpa activity_type: 0
jumlah baris: 29228
jumlah kolom: 29
+-----------+-----------------+----------+-----------+-----------+--------------+------------------+-----------------+------------------------+--------------+--------------+------------------+------------+-------------+--------------+--------------+------------------+-------------------+---------------+------------------+------------+----------+-------------------+----------+--------------------+--------------+-------------------+-------------+---------+
|code_module|code_presentation|id_student|total_click|active_days|distinct_sites|first_activity_day|last_activity_day|avg_click_per_active_day|click_dataplus|click_dualpane|click_externalquiz|click_folder|click_forumng|click_glossary|click_homepage|click_htmlactivity|click_oucollaborate|click_oucontent|click_ouelluminate|click_ouwiki|click_page|click_questionnaire|click_quiz|click_repeatactivity|click_resource|click_sharedsubpage|click_subpage|click_url|
+-----------+---

In [7]:
keys = ["code_module", "code_presentation", "id_student"]

# Gabungkan nilai siswa dengan info tugas (untuk tahu modul, tipe, dan tenggat)
assess = student_assessment.join(assessments, "id_assessment", "inner")

# Buang ujian akhir (Exam): skornya hampir identik dengan hasil akhir -> data leakage
assess = assess.filter(F.col("assessment_type") != "Exam")

# Samakan batas hari dengan fitur VLE
if CUTOFF is not None:
    assess = assess.filter(F.col("date_submitted") <= CUTOFF)

assess = (assess
    .withColumn("days_late", F.col("date_submitted") - F.col("date"))
    .withColumn("is_late", (F.col("days_late") > 0).cast("int")))

assess_agg = assess.groupBy(*keys).agg(
    F.count("*").alias("n_assessments"),
    F.round(F.avg("score"), 2).alias("avg_score"),
    F.min("score").alias("min_score"),
    F.max("score").alias("max_score"),
    F.sum("is_late").alias("n_late"),
    F.round(F.avg("days_late"), 2).alias("avg_days_late"),
    F.sum("is_banked").alias("n_banked"),
    F.round(F.avg(F.when(F.col("assessment_type") == "TMA", F.col("score"))), 2).alias("avg_score_tma"),
    F.round(F.avg(F.when(F.col("assessment_type") == "CMA", F.col("score"))), 2).alias("avg_score_cma"),
)

print("jumlah baris:", assess_agg.count())
print("jumlah kolom:", len(assess_agg.columns))
assess_agg.show(5)

# Cek null per kolom
assess_agg.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in assess_agg.columns]).show()

assess_agg.write.mode("overwrite").parquet(f"{HDFS}/processed/assess_features")

jumlah baris: 25839
jumlah kolom: 12
+-----------+-----------------+----------+-------------+---------+---------+---------+------+-------------+--------+-------------+-------------+
|code_module|code_presentation|id_student|n_assessments|avg_score|min_score|max_score|n_late|avg_days_late|n_banked|avg_score_tma|avg_score_cma|
+-----------+-----------------+----------+-------------+---------+---------+---------+------+-------------+--------+-------------+-------------+
|        AAA|            2013J|     11391|            5|     82.0|       78|       85|     0|         -1.8|       0|         82.0|         NULL|
|        AAA|            2013J|     28400|            5|     66.4|       60|       70|     2|          0.0|       0|         66.4|         NULL|
|        AAA|            2013J|     31604|            5|     76.0|       71|       88|     0|         -2.0|       0|         76.0|         NULL|
|        AAA|            2013J|     32885|            5|     54.4|       30|       75|     5|

In [8]:
keys = ["code_module", "code_presentation", "id_student"]

# Baca kembali hasil agregasi dari HDFS
vle_f    = spark.read.parquet(f"{HDFS}/processed/vle_features")
assess_f = spark.read.parquet(f"{HDFS}/processed/assess_features")

# Tabel dasar: 1 baris per siswa-modul (studentInfo)
# date_unregistration sengaja TIDAK dipakai: hanya terisi untuk siswa yang mundur -> data leakage
base = (student_info
        .join(courses, ["code_module", "code_presentation"], "left")
        .join(student_reg.select(*keys, "date_registration"), keys, "left"))

# Left join: siswa tanpa aktivitas/tugas tetap ada
ml_df = (base.join(vle_f, keys, "left")
             .join(assess_f, keys, "left"))

# Penanda "tidak ada data", dibuat SEBELUM null diisi 0
ml_df = (ml_df
    .withColumn("has_vle_activity", F.col("total_click").isNotNull().cast("int"))
    .withColumn("has_assessment", F.col("n_assessments").isNotNull().cast("int")))

# Kolom jumlah/total: tidak ada aktivitas = 0
zero_cols = (["total_click", "active_days", "distinct_sites", "avg_click_per_active_day",
              "n_assessments", "n_late", "n_banked"]
             + [c for c in ml_df.columns if c.startswith("click_")])
ml_df = ml_df.fillna(0, subset=zero_cols)

# Target: 4 kelas asli + versi biner
ml_df = ml_df.withColumn("label_binary",
    F.when(F.col("final_result").isin("Pass", "Distinction"), 1).otherwise(0))

# Verifikasi
print("jumlah baris:", ml_df.count())
print("jumlah kolom:", len(ml_df.columns))
print("kunci ganda:", ml_df.groupBy(*keys).count().filter("count > 1").count())
ml_df.groupBy("final_result").count().orderBy(F.desc("count")).show()
ml_df.groupBy("label_binary").count().show()
ml_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ml_df.columns]).show(vertical=True)

ml_df.write.mode("overwrite").parquet(f"{HDFS}/processed/features")

jumlah baris: 32593
jumlah kolom: 52
kunci ganda: 0
+------------+-----+
|final_result|count|
+------------+-----+
|        Pass|12361|
|   Withdrawn|10156|
|        Fail| 7052|
| Distinction| 3024|
+------------+-----+

+------------+-----+
|label_binary|count|
+------------+-----+
|           1|15385|
|           0|17208|
+------------+-----+

-RECORD 0---------------------------
 code_module                | 0     
 code_presentation          | 0     
 id_student                 | 0     
 gender                     | 0     
 region                     | 0     
 highest_education          | 0     
 imd_band                   | 0     
 age_band                   | 0     
 num_of_prev_attempts       | 0     
 studied_credits            | 0     
 disability                 | 0     
 final_result               | 0     
 module_presentation_length | 0     
 date_registration          | 0     
 total_click                | 0     
 active_days                | 0     
 distinct_sites        

In [10]:
chk = spark.read.parquet(f"{HDFS}/processed/features")
print("tipe date_registration:", dict(chk.dtypes)["date_registration"])
print("imd_band = '?':", chk.filter(F.col("imd_band") == "?").count())
print("date_registration = '?':", chk.filter(F.col("date_registration") == "?").count())

tipe date_registration: string
imd_band = '?': 1111
date_registration = '?': 45


In [11]:
clean = (chk
    .withColumn("imd_band",
        F.when(F.col("imd_band") == "?", None).otherwise(F.col("imd_band")))
    .withColumn("date_registration",
        F.when(F.col("date_registration") == "?", None)
         .otherwise(F.col("date_registration")).cast("int")))

clean.write.mode("overwrite").parquet(f"{HDFS}/processed/features_clean")

c2 = spark.read.parquet(f"{HDFS}/processed/features_clean")
print(c2.count(), dict(c2.dtypes)["date_registration"])
c2.select(F.sum(F.col("imd_band").isNull().cast("int")).alias("null_imd"),
          F.sum(F.col("date_registration").isNull().cast("int")).alias("null_reg")).show()

32593 int
+--------+--------+
|null_imd|null_reg|
+--------+--------+
|    1111|      45|
+--------+--------+

